## 1. Environment Setup and Artifact Loading

In [ ]:
from __future__ import annotations

import json
import os
import pickle
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
import shap
import torch

from lib.explainability import build_transaction_ae_features, load_temporal_autoencoder
from lib.model_utils import load_artifacts, load_sage_model
from lib.resource_paths import resolve_output_path

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.loader import NeighborLoader
    import torch_geometric.transforms as T
    TORCH_GEO_OK = True
except Exception:
    TORCH_GEO_OK = False

OUTPUTS_DIR = Path(resolve_output_path())
paths = {
    'rank_df': OUTPUTS_DIR / 'rank_df_with_anchor_expansion.csv.gz',
    'model_output': OUTPUTS_DIR / 'model_output.csv',
    'sage_model': OUTPUTS_DIR / 'fraud_sage_model.pth',
    'sage_artifacts': OUTPUTS_DIR / 'sage_artifacts.pkl',
    'lgbm': OUTPUTS_DIR / 'lgbm_fraud_ranker.joblib',
    'ae_scores': OUTPUTS_DIR / 'autoencoder_transaction_scores.csv.gz',
    'master_pool': OUTPUTS_DIR / 'master_transaction_pool.csv.gz',
}

missing = [k for k, p in paths.items() if not p.exists()]
assert not missing, f'Missing required artifacts: {missing}'

rank_df = pd.read_csv(paths['rank_df'])
model_output = pd.read_csv(paths['model_output'])
ae_scores = pd.read_csv(paths['ae_scores'])
master_pool = pd.read_csv(paths['master_pool'], compression='gzip', low_memory=False)

rank_df['customer_id'] = rank_df['customer_id'].astype(str)
model_output['customer_id'] = model_output['customer_id'].astype(str)
ae_scores['customer_id'] = ae_scores['customer_id'].astype(str)
master_pool['customer_id'] = master_pool['customer_id'].astype(str)

print('Artifacts loaded from:', OUTPUTS_DIR)
print('Rows -> rank_df:', len(rank_df), 'model_output:', len(model_output), 'master_pool:', len(master_pool))

## 2. Select 3 Customers by Risk Band (Low / Medium / High)

In [ ]:
risk_col = 'fraud_score' if 'fraud_score' in model_output.columns else 'scarcity_anchor_ensemble_prob'
scores = pd.to_numeric(model_output[risk_col], errors='coerce').fillna(0.0).clip(0.0, 1.0)
df = model_output[['customer_id']].copy()
df['risk_score'] = scores

low_df = df[df['risk_score'] <= 0.33].sort_values(['risk_score', 'customer_id'])
mid_df = df[(df['risk_score'] > 0.33) & (df['risk_score'] <= 0.66)].sort_values(['risk_score', 'customer_id'])
high_df = df[df['risk_score'] > 0.66].sort_values(['risk_score', 'customer_id'], ascending=[False, True])

assert len(low_df) > 0 and len(mid_df) > 0 and len(high_df) > 0, 'Could not find all risk bands.'

selected = pd.concat([
    low_df.head(1).assign(risk_band='low'),
    mid_df.head(1).assign(risk_band='medium'),
    high_df.head(1).assign(risk_band='high'),
], ignore_index=True)

selected_ids = selected['customer_id'].tolist()
selected_payload = selected.to_dict(orient='records')
sel_path = OUTPUTS_DIR / 'explainability_selected_customers.json'
sel_path.write_text(json.dumps(selected_payload, indent=2))
print('Saved:', sel_path)
display(selected)

## 3. Test LLM Endpoint Connectivity and Response Format

In [ ]:
OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://localhost:11434')
start_t = time.time()
llm_test = {'url': OLLAMA_URL, 'success': False, 'model': None, 'latency_sec': None, 'error': None, 'sample': None}

try:
    tags_r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=8)
    tags_r.raise_for_status()
    models = [m.get('name') for m in tags_r.json().get('models', []) if m.get('name')]
    assert models, 'No models found on endpoint'
    model_name = models[0]
    gen_r = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json={'model': model_name, 'prompt': 'Reply with: ok', 'stream': False, 'options': {'temperature': 0.0, 'num_predict': 20}},
        timeout=20,
    )
    gen_r.raise_for_status()
    out = gen_r.json()
    llm_test['success'] = isinstance(out, dict) and ('response' in out)
    llm_test['model'] = model_name
    llm_test['sample'] = str(out.get('response', ''))[:200]
except Exception as e:
    llm_test['error'] = str(e)

llm_test['latency_sec'] = round(time.time() - start_t, 3)
llm_test_path = OUTPUTS_DIR / 'llm_endpoint_test.json'
llm_test_path.write_text(json.dumps(llm_test, indent=2))
print('Saved:', llm_test_path)
print(json.dumps(llm_test, indent=2))

## 4. Run GNNExplainer for Selected Customers and Save Web-App-Friendly Output

In [ ]:
gnn_jsonl_path = OUTPUTS_DIR / 'gnn_explainer_webapp.jsonl'
gnn_records = []
band_map = dict(zip(selected['customer_id'], selected['risk_band']))

if TORCH_GEO_OK:
    arts = load_artifacts()
    model_sage = load_sage_model(artifacts=arts)

    data = HeteroData()
    data['customer'].x = arts['x_cust']
    data['category'].x = arts['x_cat']
    data['city'].x = arts['x_city']
    data[('customer', 'purchases_at', 'category')].edge_index = arts['edge_cust_cat']
    data[('customer', 'transacts_in', 'city')].edge_index = arts['edge_cust_city']
    data = T.ToUndirected()(data)

    cust_map = arts['cust_map']
    rev_cust = {v: k for k, v in cust_map.items()}

    class Wrap(torch.nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
        def forward(self, x_dict, edge_index_dict):
            return self.base(x_dict, edge_index_dict).squeeze(-1)

    explainer = Explainer(
        model=Wrap(model_sage).eval(),
        algorithm=GNNExplainer(epochs=15),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(mode='binary_classification', task_level='node', return_type='raw'),
    )

    for cid in selected_ids:
        rec = {'customer_id': cid, 'risk_band': band_map[cid], 'important_edges': [], 'important_nodes': [], 'mask_summary': {}, 'runtime_sec': None}
        t0 = time.time()
        try:
            idx = int(cust_map[cid])
            seed = torch.tensor([idx], dtype=torch.long)
            sub = next(iter(NeighborLoader(data, num_neighbors=[20, 15, 10], input_nodes=('customer', seed), batch_size=1, shuffle=False, num_workers=0)))
            loc = int(torch.where(sub['customer'].n_id == idx)[0][0].item())
            exp = explainer(sub.x_dict, sub.edge_index_dict, index=loc)

            for rel, mask in getattr(exp, 'edge_mask_dict', {}).items():
                if mask is None or mask.numel() == 0:
                    continue
                topk = min(5, int(mask.numel()))
                vals, pos = torch.topk(mask, k=topk)
                eidx = sub.edge_index_dict[rel]
                rel_name = '__'.join(rel)
                for v, p in zip(vals.tolist(), pos.tolist()):
                    src = int(eidx[0, p].item())
                    dst = int(eidx[1, p].item())
                    rec['important_edges'].append({'relation': rel_name, 'score': float(v), 'src_local': src, 'dst_local': dst})

            cm = getattr(exp, 'node_mask_dict', {}).get('customer')
            if cm is not None and cm.ndim == 2 and loc < cm.shape[0]:
                m = cm[loc]
                topf = torch.topk(m, k=min(5, m.numel()))
                rec['important_nodes'] = [{'feature_index': int(i), 'score': float(s)} for s, i in zip(topf.values.tolist(), topf.indices.tolist())]

            scores = [e['score'] for e in rec['important_edges']]
            rec['mask_summary'] = {
                'edge_count': len(scores),
                'edge_mean': float(np.mean(scores)) if scores else 0.0,
                'edge_max': float(np.max(scores)) if scores else 0.0,
            }
        except Exception as e:
            rec['mask_summary'] = {'error': str(e)}

        rec['runtime_sec'] = round(time.time() - t0, 3)
        gnn_records.append(rec)
else:
    for cid in selected_ids:
        gnn_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'important_edges': [],
            'important_nodes': [],
            'mask_summary': {'error': 'torch_geometric unavailable'},
            'runtime_sec': 0.0,
        })

with open(gnn_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in gnn_records:
        f.write(json.dumps(rec) + '\n')

print('Saved:', gnn_jsonl_path)
print('Records:', len(gnn_records))

## 5. Compute SHAP Values for LightGBM and Save Top Feature Attributions

In [ ]:

# ── Section 5: SHAP Values for LightGBM ──────────────────────────────────────
# The LGB model was trained with .values (no column names → Column_0..32).
# We feed it rank_df which contains all original training features.

LGB_FEATURES = [
    'emb_pca_1','emb_pca_2','emb_pca_3','emb_pca_4',
    'emb_pca_5','emb_pca_6','emb_pca_7','emb_pca_8',
    'km_component_size','km_component_train_fraud_rate','km_component_mean_dgi',
    'hdb_component_size','hdb_component_fraud_rate_labeled','hdb_component_fraud_lift_labeled',
    'knn_mean_distance','knn_suspicious_share','knn_gold_fraud_count',
    'dist_to_fraud_centroid','dist_to_legit_centroid','centroid_margin',
    'dgi_anomaly_score','customer_ae_risk_norm','gmm_max_prob','component_confidence',
    'mlp_fraud_prob','cluster_consensus_score',
    'hdb_outlier_score','min_dist_to_fraud_anchor','mean_dist_to_fraud_anchor',
    'min_dist_to_legit_anchor','anchor_proximity_score',
    'eft_amount_match_count','abm_dc_colocated',
]
LGB_FEATURES = [c for c in LGB_FEATURES if c in rank_df.columns]
assert len(LGB_FEATURES) == 33, f"Expected 33 LGB features, got {len(LGB_FEATURES)}"

rdf = rank_df.set_index('customer_id')
X = rdf.reindex(selected_ids)[LGB_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)

expl = shap.TreeExplainer(lgb_model)
shap_vals = expl.shap_values(X.values)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # class-1 (fraud) SHAP values

shap_records = []
for i, cid in enumerate(selected_ids):
    sv = shap_vals[i]
    top_idx = np.argsort(np.abs(sv))[::-1][:10]
    top_feats = [
        {"feature": LGB_FEATURES[j], "shap_value": float(sv[j]), "feature_value": float(X.values[i, j])}
        for j in top_idx
    ]
    shap_records.append({
        "customer_id": cid,
        "risk_band": band_map[cid],
        "lgb_fraud_prob": float(lgb_model.predict_proba(X.values[i:i+1])[0, 1]),
        "base_value": float(expl.expected_value if not isinstance(expl.expected_value, (list, np.ndarray)) else expl.expected_value[1]),
        "top_shap_features": top_feats,
    })

with open(shap_jsonl_path, 'w') as f:
    for r in shap_records:
        f.write(json.dumps(r) + '\n')

print(f"Saved: {shap_jsonl_path}")
print(f"Records: {len(shap_records)}")
for r in shap_records:
    top = r['top_shap_features'][0]
    print(f"  {r['customer_id']} ({r['risk_band']}) lgb_prob={r['lgb_fraud_prob']:.3f}  top_feat={top['feature']}  shap={top['shap_value']:+.3f}")


## 6. Extract Autoencoder Anomalous Transactions for Selected Customers

In [ ]:
ae_jsonl_path = OUTPUTS_DIR / 'ae_anomalies_webapp.jsonl'

tx = master_pool[master_pool['customer_id'].isin(selected_ids)].copy()
tx['transaction_datetime'] = pd.to_datetime(tx['transaction_datetime'], errors='coerce')

# Prefer saved per-transaction AE scores if transaction_id exists in both tables
if 'transaction_id' in tx.columns and 'transaction_id' in ae_scores.columns:
    ae_scores['transaction_id'] = ae_scores['transaction_id'].astype(str)
    tx['transaction_id'] = tx['transaction_id'].astype(str)
    tx = tx.merge(ae_scores[['transaction_id', 'customer_id', 'reconstruction_error', 'ae_risk_score']], on=['transaction_id', 'customer_id'], how='left')

if 'reconstruction_error' not in tx.columns:
    tx['reconstruction_error'] = np.nan
if 'ae_risk_score' not in tx.columns:
    tx['ae_risk_score'] = np.nan

ae_records = []
for cid in selected_ids:
    c = tx[tx['customer_id'] == cid].copy()
    c['reconstruction_error'] = pd.to_numeric(c['reconstruction_error'], errors='coerce')
    c['ae_risk_score'] = pd.to_numeric(c['ae_risk_score'], errors='coerce')
    c = c.sort_values(['ae_risk_score', 'reconstruction_error'], ascending=False).head(10)

    for _, r in c.iterrows():
        ae_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'transaction_id': str(r.get('transaction_id', '')),
            'transaction_datetime': str(r.get('transaction_datetime', '')),
            'amount_cad': float(pd.to_numeric(r.get('amount_cad', 0.0), errors='coerce') or 0.0),
            'merchant_category': str(r.get('merchant_category', '')),
            'city': str(r.get('city', '')),
            'source_dataset': str(r.get('source_dataset', '')),
            'cash_indicator': int(pd.to_numeric(r.get('cash_indicator', 0), errors='coerce') or 0),
            'ecommerce_ind': int(pd.to_numeric(r.get('ecommerce_ind', 0), errors='coerce') or 0),
            'reconstruction_error': float(pd.to_numeric(r.get('reconstruction_error', 0.0), errors='coerce') or 0.0),
            'ae_risk_score': float(pd.to_numeric(r.get('ae_risk_score', 0.0), errors='coerce') or 0.0),
        })

with open(ae_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in ae_records:
        f.write(json.dumps(rec) + '\n')

print('Saved:', ae_jsonl_path)
print('Rows:', len(ae_records))

## 7. Batch LLM Explanations for GNN + SHAP + AE Evidence

In [ ]:

# ── Section 7: Batch LLM Explanations ────────────────────────────────────────
explain_jsonl_path = OUTPUTS_DIR / 'customer_explanations_webapp.jsonl'

gnn_by_cid  = {r['customer_id']: r for r in gnn_records}
shap_by_cid = {r['customer_id']: r for r in shap_records}
ae_by_cid   = {}
for r in ae_records:
    ae_by_cid.setdefault(r['customer_id'], []).append(r)

def fallback_text(cid, band, pred, top_shap, top_ae):
    return (
        f"Customer {cid} is in the {band} risk band with model score {pred:.3f}. "
        f"Key model drivers include {top_shap}. "
        f"The strongest anomalous transaction pattern is {top_ae}. "
        "Review this customer using the combined graph, feature-attribution, and transaction evidence."
    )

def call_llm(prompt):
    if not llm_test.get('success'):
        raise RuntimeError('endpoint unavailable')
    resp = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            'model': llm_test['model'],
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': 0.2, 'num_predict': 220},
        },
        timeout=45,
    )
    resp.raise_for_status()
    return str(resp.json().get('response', '')).strip()

explanation_records = []
for cid in selected_ids:
    band = band_map[cid]
    s = shap_by_cid.get(cid, {})
    g = gnn_by_cid.get(cid, {})
    a = ae_by_cid.get(cid, [])

    # SHAP record uses 'lgb_fraud_prob' and 'top_shap_features'
    pred = float(s.get('lgb_fraud_prob', s.get('prediction', 0.0)))
    top_shap_items = s.get('top_shap_features', s.get('top_features', []))[:3]
    top_shap_text = '; '.join(
        [f"{x['feature']} ({float(x.get('shap_value', x.get('value', 0))):+.4f})" for x in top_shap_items]
    ) or 'no SHAP drivers'

    # GNN record
    top_edges = g.get('important_edges', [])[:3]
    top_edges_text = (
        '; '.join([f"{x['relation']} score={x['score']:.3f}" for x in top_edges])
        or 'no significant graph edges'
    )

    top_ae = a[0] if a else None
    top_ae_text = (
        f"{top_ae.get('merchant_category','?')} in {top_ae.get('city','?')} "
        f"amount ${float(top_ae.get('amount_cad', 0) or 0):.2f} "
        f"ae={float(top_ae.get('ae_risk_score', top_ae.get('reconstruction_error', 0)) or 0):.4f}"
        if top_ae else 'no high-anomaly transaction found'
    )

    prompt = (
        "Write a concise investigator-facing explanation in 4-6 sentences.\n"
        f"Customer: {cid}\n"
        f"Risk band: {band}\n"
        f"Model fraud probability: {pred:.4f}\n"
        f"Top SHAP drivers: {top_shap_text}\n"
        f"Graph (GNN) evidence: {top_edges_text}\n"
        f"Autoencoder anomalies: {top_ae_text}\n"
        "Mention the most important checks the analyst should do next."
    )

    llm_status = 'ok'
    explanation_text = ''
    for attempt in range(3):
        try:
            explanation_text = call_llm(prompt)
            if explanation_text:
                break
        except Exception as e:
            llm_status = f"error_attempt_{attempt + 1}: {e}"
            time.sleep(0.5 * (attempt + 1))

    if not explanation_text:
        llm_status = 'fallback'
        explanation_text = fallback_text(cid, band, pred, top_shap_text, top_ae_text)

    explanation_records.append({
        'customer_id': cid,
        'risk_band': band,
        'model_fraud_prob': pred,
        'explanation_text': explanation_text,
        'llm_status': llm_status,
        'prompt_used': prompt,
    })

with open(explain_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in explanation_records:
        f.write(json.dumps(rec) + '\n')

print(f"Saved: {explain_jsonl_path}")
print(f"Records: {len(explanation_records)}")
for rec in explanation_records:
    print(f"\n{'='*60}")
    print(f"Customer: {rec['customer_id']}  ({rec['risk_band']})  LLM: {rec['llm_status']}")
    print(rec['explanation_text'])


## 8. Write Unified Web App Explanation Files and Verification Preview

In [ ]:

# ── Section 8: Write Unified Web App Explanation Files ───────────────────────
bundle_jsonl_path = OUTPUTS_DIR / 'explainability_bundle_webapp.jsonl'
bundle_csv_path   = OUTPUTS_DIR / 'explainability_bundle_webapp.csv'

# Re-read all JSONL outputs to ensure consistent data
def _read_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]

shap_data = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'lgbm_shap_webapp.jsonl')}
ae_data   = {}
for r in _read_jsonl(OUTPUTS_DIR / 'ae_anomalies_webapp.jsonl'):
    ae_data.setdefault(r['customer_id'], []).append(r)
exp_data  = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'customer_explanations_webapp.jsonl')}
gnn_data  = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'gnn_explainer_webapp.jsonl')}

# Risk scores from the original model output
sel_scores = selected.set_index('customer_id')['risk_score'].to_dict()

rows = []
for cid in selected_ids:
    band = band_map[cid]

    s = shap_data.get(cid, {})
    risk_score = s.get('lgb_fraud_prob', sel_scores.get(cid, 0.0))
    top_shap_feats = s.get('top_shap_features', [])
    top_shap_name  = top_shap_feats[0]['feature'] if top_shap_feats else ''

    ae_list  = ae_data.get(cid, [])
    top_txn  = ae_list[0] if ae_list else {}

    c_exp = exp_data.get(cid, {})

    row = {
        'customer_id': cid,
        'risk_band': band,
        'risk_score': float(risk_score),
        'top_shap_feature': top_shap_name,
        'top_anomalous_transaction_id': str(top_txn.get('transaction_id', '')),
        'top_anomalous_amount_cad': float(top_txn.get('amount_cad', 0.0) or 0.0),
        'explanation_text': c_exp.get('explanation_text', ''),
        'llm_status': c_exp.get('llm_status', ''),
    }
    rows.append(row)

bundle_df = pd.DataFrame(rows)
bundle_df.to_csv(bundle_csv_path, index=False)

with open(bundle_jsonl_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')

print('Saved:', bundle_jsonl_path)
print('Saved:', bundle_csv_path)
print(f'\nVerification preview ({len(rows)} customers):')
display(bundle_df[['customer_id', 'risk_band', 'risk_score', 'top_shap_feature', 'top_anomalous_transaction_id', 'explanation_text']])
